# Part 2 — WLASL Word-Level ASL Recognition
## Exploratory Data Analysis

This notebook is split into two phases:
- **Phase A** (before preprocessing): Analyse the raw JSON metadata and download results.
- **Phase B** (after preprocessing): Analyse the extracted landmark sequences.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter, defaultdict

# Paths — adjust if running from a different directory
DATA_DIR     = os.path.join("..", "data")
WLASL_JSON   = os.path.join(DATA_DIR, "WLASL_v0.3.json")
AVAIL_JSON   = os.path.join(DATA_DIR, "available_videos.json")
FAILED_LOG   = os.path.join(DATA_DIR, "failed_downloads.txt")
SEQ_DIR      = os.path.join(DATA_DIR, "sequences")
RESULTS_DIR  = os.path.join("..", "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded.")

---
## Phase A — Raw Dataset Analysis
### A1. Load WLASL JSON and inspect structure

In [ ]:
with open(WLASL_JSON) as f:
    wlasl_data = json.load(f)

print(f"Total glosses in full WLASL dataset: {len(wlasl_data)}")
print("\nExample entry (first gloss):")
print(json.dumps(wlasl_data[0], indent=2)[:800])

In [ ]:
# Flatten all instances into a DataFrame for easy analysis
rows = []
for entry in wlasl_data:
    gloss = entry["gloss"]
    for inst in entry.get("instances", []):
        rows.append({
            "gloss":       gloss,
            "video_id":    str(inst.get("video_id", "")).zfill(5),
            "split":       inst.get("split", "train"),
            "signer_id":   inst.get("signer_id", -1),
            "frame_start": inst.get("frame_start", 1),
            "frame_end":   inst.get("frame_end", -1),
            "url":         inst.get("url", ""),
        })

df_full = pd.DataFrame(rows)
print(f"Total instances in full WLASL: {len(df_full)}")
df_full.head()

### A2. WLASL100 — Instance count distribution

In [ ]:
# Sort glosses by instance count, take top 100
gloss_counts = df_full.groupby("gloss").size().sort_values(ascending=False)
wlasl100_glosses = gloss_counts.head(100)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: top-100 instance counts per gloss
axes[0].bar(range(100), wlasl100_glosses.values, color="steelblue", width=1.0)
axes[0].set_xlabel("Gloss rank (0 = most instances)")
axes[0].set_ylabel("Number of video instances")
axes[0].set_title("WLASL100: instances per gloss")

# Right: histogram of instance counts
axes[1].hist(wlasl100_glosses.values, bins=20, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Instance count")
axes[1].set_ylabel("Number of glosses")
axes[1].set_title("Distribution of instance counts (WLASL100)")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_raw_wlasl100.png"), bbox_inches="tight")
plt.show()

print(f"Min instances: {wlasl100_glosses.min()}")
print(f"Max instances: {wlasl100_glosses.max()}")
print(f"Mean instances: {wlasl100_glosses.mean():.1f}")
print(f"\nTop 10 glosses:\n{wlasl100_glosses.head(10).to_string()}")

### A3. Split distribution (train / val / test)

In [ ]:
df_100 = df_full[df_full["gloss"].isin(wlasl100_glosses.index)].copy()
split_counts = df_100["split"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
split_counts.plot.bar(ax=ax, color=["#2196F3", "#FF9800", "#4CAF50"], edgecolor="white")
ax.set_title("WLASL100: train/val/test split counts")
ax.set_xlabel("Split")
ax.set_ylabel("Number of instances")
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+p.get_width()/2, p.get_height()+2),
                ha="center", fontsize=11)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_split_distribution.png"), bbox_inches="tight")
plt.show()
print(split_counts.to_string())

### A4. Signer diversity

In [ ]:
# Number of unique signers per gloss
signers_per_gloss = df_100.groupby("gloss")["signer_id"].nunique().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
signers_per_gloss.hist(bins=range(1, signers_per_gloss.max()+2), ax=ax,
                        color="steelblue", edgecolor="white")
ax.set_title("Unique signers per gloss (WLASL100)")
ax.set_xlabel("Number of unique signers")
ax.set_ylabel("Number of glosses")
plt.tight_layout()
plt.show()

print(f"Total unique signers: {df_100['signer_id'].nunique()}")
print(f"Glosses with only 1 signer: {(signers_per_gloss == 1).sum()}")

### A5. Video duration analysis (from frame_start / frame_end)

In [ ]:
# Approximate duration in frames (assumes 25 fps)
df_100 = df_100.copy()
df_100["duration_frames"] = df_100["frame_end"] - df_100["frame_start"] + 1
df_100 = df_100[df_100["duration_frames"] > 0]   # drop invalid
df_100["duration_sec"] = df_100["duration_frames"] / 25.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_100["duration_frames"], bins=40, color="coral", edgecolor="white")
axes[0].set_title("Duration distribution (frames)")
axes[0].set_xlabel("Frames")
axes[0].set_ylabel("Count")

axes[1].hist(df_100["duration_sec"], bins=40, color="coral", edgecolor="white")
axes[1].set_title("Duration distribution (seconds @ 25 fps)")
axes[1].set_xlabel("Seconds")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_duration_distribution.png"), bbox_inches="tight")
plt.show()

print(df_100["duration_frames"].describe())

### A6. Download coverage (run after download_dataset.py)

In [ ]:
if not os.path.exists(AVAIL_JSON):
    print("available_videos.json not found — run download_dataset.py first.")
else:
    with open(AVAIL_JSON) as f:
        available = json.load(f)

    avail_ids = {r["video_id"] for r in available}
    total_ids = set(df_100["video_id"].tolist())
    failed_ids = total_ids - avail_ids

    print(f"WLASL100 total   : {len(total_ids)}")
    print(f"Downloaded OK    : {len(avail_ids)}  ({100*len(avail_ids)/len(total_ids):.1f}%)")
    print(f"Failed / missing : {len(failed_ids)}  ({100*len(failed_ids)/len(total_ids):.1f}%)")

    # Coverage per gloss
    df_avail = pd.DataFrame(available)
    coverage = df_avail.groupby("gloss").size() / df_100.groupby("gloss").size()
    coverage = coverage.fillna(0)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(coverage.values * 100, bins=20, color="#4CAF50", edgecolor="white")
    ax.set_xlabel("Download coverage (%)")
    ax.set_ylabel("Number of glosses")
    ax.set_title("Per-gloss download coverage (WLASL100)")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "dataset_download_coverage.png"), bbox_inches="tight")
    plt.show()

    # Glosses with very low coverage
    low = coverage[coverage < 0.5].sort_values()
    if len(low):
        print(f"\nGlosses with <50% coverage ({len(low)}):\n{low.to_string()}")

---
## Phase B — Preprocessed Sequences Analysis
### B1. Load sequences (run after preprocessing.py)

In [ ]:
x_path = os.path.join(SEQ_DIR, "X.npy")
if not os.path.exists(x_path):
    print("X.npy not found — run preprocessing.py first.")
else:
    X         = np.load(x_path)                                              # (N, 30, 225)
    y         = np.load(os.path.join(SEQ_DIR, "y.npy"))                      # (N,)
    label_map = np.load(os.path.join(SEQ_DIR, "label_map.npy"),
                        allow_pickle=True).item()                             # {idx: word}
    splits    = np.load(os.path.join(SEQ_DIR, "splits.npy"),
                        allow_pickle=True).item()                             # {video_id: split}
    video_ids = np.load(os.path.join(SEQ_DIR, "video_ids.npy"),
                        allow_pickle=True)                                    # (N,)

    print(f"X shape     : {X.shape}   (N, seq_len, feature_dim)")
    print(f"y shape     : {y.shape}")
    print(f"Num classes : {len(label_map)}")
    print(f"\nFirst 10 classes: {dict(list(label_map.items())[:10])}")

### B2. Samples per class (post-preprocessing)

In [ ]:
unique_classes, counts = np.unique(y, return_counts=True)
class_names = [label_map[c] for c in unique_classes]

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(class_names, counts, color="steelblue")
ax.set_xlabel("Gloss")
ax.set_ylabel("Number of sequences")
ax.set_title("Samples per class after preprocessing")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_post_preprocessing.png"), bbox_inches="tight")
plt.show()

print(f"Min: {counts.min()} ({class_names[counts.argmin()]})")
print(f"Max: {counts.max()} ({class_names[counts.argmax()]})")
print(f"Mean: {counts.mean():.1f}")

### B3. Zero-detection frames (missing hand/pose)

In [ ]:
# For each sequence, count how many of the 30 frames are all-zero (no detection)
zero_mask = (X.sum(axis=2) == 0)            # (N, 30) — True where frame is all zeros
zero_per_seq = zero_mask.sum(axis=1)         # (N,) — zero frame count per sequence

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(zero_per_seq, bins=range(0, 31), color="#FF9800", edgecolor="white")
axes[0].set_xlabel("Zero-detection frames per sequence")
axes[0].set_ylabel("Number of sequences")
axes[0].set_title("Zero-frame distribution (all sequences)")

# Per-feature zero fraction across the whole dataset
zero_frac_per_feat = (X == 0).mean(axis=(0, 1))   # (225,)
axes[1].plot(zero_frac_per_feat, linewidth=0.8, color="steelblue")
axes[1].axvline(63,  color="red",   linestyle="--", linewidth=0.8, label="LH end")
axes[1].axvline(126, color="green", linestyle="--", linewidth=0.8, label="RH end")
axes[1].set_xlabel("Feature dimension")
axes[1].set_ylabel("Fraction of zeros")
axes[1].set_title("Zero-value fraction per feature")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_zero_detection.png"), bbox_inches="tight")
plt.show()

fully_zero = (zero_per_seq == 30).sum()
print(f"Sequences with ALL frames zero (no detection at all): {fully_zero}")
print(f"Mean zero-frames per sequence: {zero_per_seq.mean():.2f} / 30")

### B4. Temporal feature profiles for sample words

In [ ]:
# Pick 6 sample words and plot the x-coordinate of the right wrist over 30 frames
# Right hand starts at feature index 63; landmark 0 (wrist) x-coord is index 63+0=63
RH_WRIST_X = 63   # index in 225-D feature vector

sample_classes = unique_classes[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)

for ax, cls in zip(axes.flat, sample_classes):
    cls_indices = np.where(y == cls)[0]
    for i in cls_indices[:5]:   # up to 5 examples
        ax.plot(X[i, :, RH_WRIST_X], alpha=0.6, linewidth=1)
    ax.set_title(label_map[cls], fontsize=12)
    ax.set_xlabel("Frame (0→29)")
    ax.set_ylabel("RH wrist x")

plt.suptitle("Right-hand wrist x-coordinate over time (5 examples each)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_temporal_profiles.png"), bbox_inches="tight")
plt.show()

### B5. Feature statistics (mean & std across dataset)

In [ ]:
feat_mean = X.mean(axis=(0, 1))   # (225,)
feat_std  = X.std(axis=(0, 1))    # (225,)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(feat_mean, linewidth=0.8, color="steelblue")
axes[0].axvline(63,  color="red",   linestyle="--", linewidth=0.8, label="LH/RH boundary")
axes[0].axvline(126, color="green", linestyle="--", linewidth=0.8, label="RH/Pose boundary")
axes[0].set_ylabel("Mean")
axes[0].set_title("Feature mean across all frames & sequences")
axes[0].legend()

axes[1].plot(feat_std, linewidth=0.8, color="coral")
axes[1].axvline(63,  color="red",   linestyle="--", linewidth=0.8)
axes[1].axvline(126, color="green", linestyle="--", linewidth=0.8)
axes[1].set_ylabel("Std")
axes[1].set_xlabel("Feature dimension")
axes[1].set_title("Feature std across all frames & sequences")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_feature_statistics.png"), bbox_inches="tight")
plt.show()

### B6. Train / Val / Test split verification

In [ ]:
split_labels = [splits.get(vid, "train") for vid in video_ids]
split_counter = Counter(split_labels)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(split_counter.keys(), split_counter.values(),
              color=["#2196F3", "#FF9800", "#4CAF50"][:len(split_counter)])
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha="center")
ax.set_title("Sequence counts by split (post-preprocessing)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "dataset_split_counts.png"), bbox_inches="tight")
plt.show()
print(split_counter)

### B7. Summary

In [ ]:
train_idx = [i for i, vid in enumerate(video_ids) if splits.get(vid, "train") == "train"]
val_idx   = [i for i, vid in enumerate(video_ids) if splits.get(vid, "train") == "val"]
test_idx  = [i for i, vid in enumerate(video_ids) if splits.get(vid, "train") == "test"]

print("=" * 50)
print("  DATASET SUMMARY")
print("=" * 50)
print(f"  Total sequences  : {len(X)}")
print(f"  Sequence shape   : {X.shape[1:]}  (seq_len × feature_dim)")
print(f"  Num classes      : {len(label_map)}")
print(f"  Train sequences  : {len(train_idx)}")
print(f"  Val sequences    : {len(val_idx)}")
print(f"  Test sequences   : {len(test_idx)}")
print(f"  Zero-frame rate  : {zero_per_seq.mean():.2f} / 30 frames avg")
print(f"  Samples/class    : min={counts.min()}, max={counts.max()}, mean={counts.mean():.1f}")
print("=" * 50)
print("\nReady to train! Next step: run train.py (on Colab with GPU recommended).")